# 🏈 Football Detection Performance Evaluation

## 📋 Table of Contents

### Part 1: Setup & Data Loading
- [1.1 Environment Setup](#setup)
- [1.2 Data Loading](#data-loading)  
- [1.3 Quick Overview](#overview)

### Part 2: Core Analysis
- [2.1 Detection Overview Statistics](#stats)
- [2.2 Confidence Score Analysis](#confidence)
- [2.3 Temporal Pattern Analysis](#temporal)
- [2.4 Spatial Distribution Analysis](#spatial)

### Part 3: Performance Evaluation
- [3.1 Performance Metrics Calculation](#metrics)
- [3.2 Results Interpretation](#interpretation)
- [3.3 Recommendations](#recommendations)

### Part 4: Reference & Troubleshooting
- [4.1 Metrics Guide](#guide)
- [4.2 Quick Reference](#reference)

---

## 🎯 What This Notebook Does

This comprehensive evaluation analyzes your football object detection system across multiple dimensions:

- **📊 Detection Quality**: Confidence scores, detection counts, consistency
- **⏱️ Temporal Patterns**: How performance varies over time
- **🗺️ Spatial Distribution**: Where objects are detected on the field  
- **🎯 Performance Scoring**: Standardized metrics with actionable insights
- **💡 Recommendations**: Specific improvements based on your data

**Prerequisites**: Run `python demo_pipeline.py` to generate detection results first.

# Part 1: Setup & Data Loading

<a id="setup"></a>
## 1.1 🔧 Environment Setup

Setting up the analysis environment with required libraries and configurations.

In [ ]:
# Import Required Libraries
import sys
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, Counter
from typing import Dict, List, Tuple, Any
import warnings
warnings.filterwarnings('ignore')

# Add the parent directory to the path
sys.path.append("/workspaces/football_analysis")

# Set up plotting style for better visualizations
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("✅ Libraries imported successfully!")
print(f"📁 Current working directory: {os.getcwd()}")
print(f"🐍 Python path includes: {sys.path[-1]}")
print(f"📊 Matplotlib backend: {plt.get_backend()}")
print(f"🎨 Seaborn version: {sns.__version__}")

<a id="data-loading"></a>
## 1.2 📂 Data Loading

Loading detection results from the pipeline and preparing data for analysis.

In [ ]:
# Data Loading and Preparation Functions
def load_detection_data():
    """Load detection results from the pipeline output."""
    results_path = "/workspaces/football_analysis/outputs/data/pipeline_results.pkl"
    
    if not os.path.exists(results_path):
        print("❌ No pipeline results found!")
        print("   Please run the pipeline first: python demo_pipeline.py")
        return None
    
    print("📊 Loading detection data...")
    with open(results_path, "rb") as f:
        data = pickle.load(f)
    
    # Extract frames data
    if isinstance(data, dict) and "video_data" in data:
        video_data = data["video_data"]
        frames = video_data.frames if hasattr(video_data, "frames") else []
    else:
        frames = (
            data.get("frames", [])
            if isinstance(data, dict)
            else getattr(data, "frames", [])
        )
    
    print(f"✅ Loaded {len(frames)} frames of detection data")
    return frames

def extract_detection_metrics(frames):
    """Extract comprehensive detection metrics from frames."""
    if not frames:
        print("❌ No frames data provided")
        return None
        
    metrics = {
        'frame_detections': [],
        'all_detections': [],
        'confidence_scores': [],
        'object_types': [],
        'bbox_areas': [],
        'bbox_centers': [],
        'frame_numbers': [],
        'detection_counts_by_type': defaultdict(list),
        'spatial_distribution': defaultdict(list)
    }
    
    print("🔍 Extracting detection metrics...")
    
    for frame_idx, frame_data in enumerate(frames):
        detections = getattr(frame_data, "detections", []) or []
        frame_detection_count = len(detections)
        metrics['frame_detections'].append(frame_detection_count)
        
        frame_type_counts = Counter()
        
        for detection in detections:
            # Basic detection info
            confidence = getattr(detection, "confidence", 0.0)
            if hasattr(detection, "bbox") and hasattr(detection.bbox, "confidence"):
                confidence = detection.bbox.confidence
            
            object_type = getattr(detection, "object_type", "unknown")
            object_type_str = str(object_type).split('.')[-1].lower() if hasattr(object_type, '__class__') else str(object_type)
            
            metrics['all_detections'].append({
                'frame_idx': frame_idx,
                'confidence': confidence,
                'object_type': object_type_str,
                'detection': detection
            })
            
            metrics['confidence_scores'].append(confidence)
            metrics['object_types'].append(object_type_str)
            frame_type_counts[object_type_str] += 1
            
            # Spatial metrics
            if hasattr(detection, "bbox"):
                bbox = detection.bbox
                width = bbox.x2 - bbox.x1
                height = bbox.y2 - bbox.y1
                area = width * height
                center_x = (bbox.x1 + bbox.x2) / 2
                center_y = (bbox.y1 + bbox.y2) / 2
                
                metrics['bbox_areas'].append(area)
                metrics['bbox_centers'].append((center_x, center_y))
                metrics['spatial_distribution'][object_type_str].append((center_x, center_y))
            
            metrics['frame_numbers'].append(frame_idx)
        
        # Store per-frame type counts
        for obj_type in ['player', 'goalkeeper', 'referee', 'ball']:
            metrics['detection_counts_by_type'][obj_type].append(frame_type_counts[obj_type])
    
    print(f"✅ Extracted metrics for {len(metrics['all_detections'])} total detections")
    return metrics

# Load the data
print("🚀 Starting data loading process...")
frames_data = load_detection_data()

if frames_data:
    detection_metrics = extract_detection_metrics(frames_data)
    if detection_metrics:
        print("🎯 Detection metrics extraction complete!")
        print(f"📊 Ready to analyze {len(frames_data)} frames with {len(detection_metrics['all_detections'])} detections")
    else:
        print("❌ Failed to extract detection metrics")
else:
    print("❌ Failed to load detection data")

<a id="overview"></a>
## 1.3 👀 Quick Overview

A high-level summary of your detection data before diving into detailed analysis.

In [ ]:
# Quick Overview Analysis
if frames_data and detection_metrics:
    total_frames = len(frames_data)
    total_detections = len(detection_metrics['all_detections'])
    
    print("🎯 DETECTION SYSTEM OVERVIEW")
    print("=" * 50)
    print(f"📹 Total Frames: {total_frames:,}")
    print(f"🎯 Total Detections: {total_detections:,}")
    print(f"📊 Avg Detections/Frame: {total_detections/total_frames:.1f}")
    
    # Object type summary
    object_counts = Counter(detection_metrics['object_types'])
    print(f"\n🏷️  Object Types Detected:")
    for obj_type, count in object_counts.most_common():
        percentage = (count / total_detections) * 100
        print(f"   {obj_type.capitalize()}: {count:,} ({percentage:.1f}%)")
    
    # Confidence summary
    confidences = detection_metrics['confidence_scores']
    if confidences:
        print(f"\n🎲 Confidence Summary:")
        print(f"   Mean: {np.mean(confidences):.3f}")
        print(f"   Range: {np.min(confidences):.3f} - {np.max(confidences):.3f}")
        
        high_conf = sum(1 for c in confidences if c > 0.8)
        low_conf = sum(1 for c in confidences if c < 0.3)
        print(f"   High confidence (>0.8): {high_conf:,} ({high_conf/len(confidences)*100:.1f}%)")
        print(f"   Low confidence (<0.3): {low_conf:,} ({low_conf/len(confidences)*100:.1f}%)")
    
    # Frame coverage
    frames_with_detections = sum(1 for count in detection_metrics['frame_detections'] if count > 0)
    coverage = frames_with_detections / total_frames * 100
    print(f"\n📹 Frame Coverage: {frames_with_detections:,}/{total_frames:,} frames ({coverage:.1f}%)")
    
    print(f"\n✅ Data loaded successfully - Ready for detailed analysis!")
    
else:
    print("❌ No data available for overview analysis")
    print("   Please check the data loading step above")

# Part 2: Core Analysis

<a id="stats"></a>
## 2.1 📊 Detection Overview Statistics

Comprehensive statistics about detection counts, object types, and basic performance indicators.

### 📈 What These Statistics Tell Us:
- **Detection Volume**: How many objects are detected per frame
- **Object Distribution**: Balance between players, ball, referees, etc.
- **Coverage**: Percentage of frames with successful detections
- **Confidence Health**: Overall model certainty levels

In [ ]:
# Detailed Detection Statistics
if frames_data and detection_metrics:
    print("📊 DETAILED DETECTION STATISTICS")
    print("=" * 60)
    
    total_frames = len(frames_data)
    total_detections = len(detection_metrics['all_detections'])
    frame_detections = detection_metrics['frame_detections']
    confidences = detection_metrics['confidence_scores']
    
    # Frame-level analysis
    print(f"📹 FRAME ANALYSIS:")
    print(f"   Total frames processed: {total_frames:,}")
    frames_with_detections = sum(1 for count in frame_detections if count > 0)
    print(f"   Frames with detections: {frames_with_detections:,} ({frames_with_detections/total_frames*100:.1f}%)")
    print(f"   Empty frames: {total_frames - frames_with_detections:,} ({(total_frames - frames_with_detections)/total_frames*100:.1f}%)")
    
    # Detection count analysis
    if frame_detections:
        avg_detections = np.mean(frame_detections)
        std_detections = np.std(frame_detections)
        max_detections = max(frame_detections)
        min_detections = min(frame_detections)
        
        print(f"\n🎯 DETECTION COUNT ANALYSIS:")
        print(f"   Total detections: {total_detections:,}")
        print(f"   Average per frame: {avg_detections:.2f} ± {std_detections:.2f}")
        print(f"   Range: {min_detections} - {max_detections} per frame")
        print(f"   Median: {np.median(frame_detections):.1f}")
    
    # Object type distribution
    object_type_counts = Counter(detection_metrics['object_types'])
    print(f"\n🏷️  OBJECT TYPE DISTRIBUTION:")
    for obj_type, count in object_type_counts.most_common():
        percentage = count / total_detections * 100
        print(f"   {obj_type.capitalize()}: {count:,} ({percentage:.1f}%)")
    
    # Confidence analysis
    if confidences:
        print(f"\n🎲 CONFIDENCE SCORE ANALYSIS:")
        print(f"   Mean confidence: {np.mean(confidences):.3f}")
        print(f"   Median confidence: {np.median(confidences):.3f}")
        print(f"   Standard deviation: {np.std(confidences):.3f}")
        print(f"   Range: {np.min(confidences):.3f} - {np.max(confidences):.3f}")
        
        # Confidence categories
        high_conf = sum(1 for conf in confidences if conf > 0.8)
        medium_conf = sum(1 for conf in confidences if 0.5 <= conf <= 0.8)
        low_conf = sum(1 for conf in confidences if conf < 0.5)
        very_low_conf = sum(1 for conf in confidences if conf < 0.3)
        
        print(f"\n   📈 CONFIDENCE CATEGORIES:")
        print(f"      High (>0.8): {high_conf:,} ({high_conf/len(confidences)*100:.1f}%)")
        print(f"      Medium (0.5-0.8): {medium_conf:,} ({medium_conf/len(confidences)*100:.1f}%)")
        print(f"      Low (<0.5): {low_conf:,} ({low_conf/len(confidences)*100:.1f}%)")
        print(f"      Very Low (<0.3): {very_low_conf:,} ({very_low_conf/len(confidences)*100:.1f}%)")
    
    print(f"\n✅ Statistics analysis complete!")
    
else:
    print("❌ No data available for statistical analysis")

<a id="confidence"></a>
## 2.2 📈 Confidence Score Analysis

Understanding confidence score patterns helps optimize detection thresholds and identify model reliability.

### 📊 Visualization Guide:
- **Histogram**: Shows distribution shape (ideal: right-skewed peak around 0.7-0.9)
- **Box Plots**: Compare confidence across object types
- **Cumulative Plot**: Shows threshold impact (how many detections you'd keep)
- **Per-Type Analysis**: Different objects may have different confidence patterns

In [ ]:
# Confidence Score Visualization and Analysis
if frames_data and detection_metrics:
    confidences = detection_metrics['confidence_scores']
    object_types = detection_metrics['object_types']
    
    if confidences:
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('Confidence Score Analysis', fontsize=16, fontweight='bold')
        
        # 1. Overall confidence distribution
        axes[0, 0].hist(confidences, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
        mean_conf = np.mean(confidences)
        median_conf = np.median(confidences)
        axes[0, 0].axvline(mean_conf, color='red', linestyle='--', linewidth=2,
                           label=f'Mean: {mean_conf:.3f}')
        axes[0, 0].axvline(median_conf, color='orange', linestyle='--', linewidth=2,
                           label=f'Median: {median_conf:.3f}')
        axes[0, 0].set_xlabel('Confidence Score')
        axes[0, 0].set_ylabel('Frequency')
        axes[0, 0].set_title('Overall Confidence Distribution')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        
        # 2. Confidence by object type (histogram)
        unique_types = list(set(object_types))
        colors = plt.cm.Set3(np.linspace(0, 1, len(unique_types)))
        
        for i, obj_type in enumerate(unique_types):
            type_confidences = [conf for conf, otype in zip(confidences, object_types) if otype == obj_type]
            if type_confidences:
                axes[0, 1].hist(type_confidences, bins=20, alpha=0.6, 
                               color=colors[i], label=f'{obj_type.capitalize()} (n={len(type_confidences)})',
                               edgecolor='black', linewidth=0.5)
        
        axes[0, 1].set_xlabel('Confidence Score')
        axes[0, 1].set_ylabel('Frequency')
        axes[0, 1].set_title('Confidence Distribution by Object Type')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # 3. Box plot by object type
        conf_by_type = {}
        for obj_type in unique_types:
            conf_by_type[obj_type] = [conf for conf, otype in zip(confidences, object_types) if otype == obj_type]
        
        box_data = [conf_by_type[obj_type] for obj_type in unique_types if conf_by_type[obj_type]]
        box_labels = [obj_type.capitalize() for obj_type in unique_types if conf_by_type[obj_type]]
        
        bp = axes[1, 0].boxplot(box_data, labels=box_labels, patch_artist=True)
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
        axes[1, 0].set_ylabel('Confidence Score')
        axes[1, 0].set_title('Confidence Distribution by Object Type (Box Plot)')
        axes[1, 0].tick_params(axis='x', rotation=45)
        axes[1, 0].grid(True, alpha=0.3)
        
        # 4. Cumulative distribution with thresholds
        sorted_conf = np.sort(confidences)
        cumulative = np.arange(1, len(sorted_conf) + 1) / len(sorted_conf)
        axes[1, 1].plot(sorted_conf, cumulative, linewidth=3, color='green', label='Cumulative Distribution')
        
        # Add threshold lines
        thresholds = [0.3, 0.5, 0.7, 0.9]
        colors_thresh = ['red', 'orange', 'blue', 'purple']
        for threshold, color in zip(thresholds, colors_thresh):
            kept_percentage = (np.sum(np.array(confidences) >= threshold) / len(confidences)) * 100
            axes[1, 1].axvline(threshold, color=color, linestyle='--', alpha=0.7, 
                               label=f'{threshold}: {kept_percentage:.1f}% kept')
        
        axes[1, 1].set_xlabel('Confidence Score')
        axes[1, 1].set_ylabel('Cumulative Probability')
        axes[1, 1].set_title('Cumulative Confidence Distribution')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Threshold analysis summary
        print("🎚️  CONFIDENCE THRESHOLD ANALYSIS:")
        thresholds = [0.1, 0.3, 0.5, 0.7, 0.8, 0.9]
        for threshold in thresholds:
            remaining = sum(1 for conf in confidences if conf >= threshold)
            percentage = (remaining / len(confidences)) * 100
            print(f"   Threshold {threshold}: {remaining:,} detections remaining ({percentage:.1f}%)")
        
        # Per object type confidence summary
        print(f"\n📊 CONFIDENCE BY OBJECT TYPE:")
        for obj_type in unique_types:
            type_confs = [conf for conf, otype in zip(confidences, object_types) if otype == obj_type]
            if type_confs:
                mean_conf = np.mean(type_confs)
                std_conf = np.std(type_confs)
                print(f"   {obj_type.capitalize()}: {mean_conf:.3f} ± {std_conf:.3f} (n={len(type_confs)})")
        
    else:
        print("❌ No confidence data available for analysis")
else:
    print("❌ No data available for confidence analysis")

<a id="temporal"></a>
## 2.3 ⏱️ Temporal Pattern Analysis

Analyzing how detection performance changes over time reveals stability issues and quality trends.

### 📈 What to Look For:
- **Stable Lines**: Good consistent performance
- **Trends**: Gradual changes may indicate lighting/quality shifts  
- **Spikes**: Sudden increases could be false positives
- **Drops**: Sudden decreases may indicate detection failures
- **Variance**: High variation suggests instability

In [ ]:
# Temporal Pattern Analysis
if frames_data and detection_metrics:
    frame_detections = detection_metrics['frame_detections']
    detection_counts_by_type = detection_metrics['detection_counts_by_type']
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Temporal Detection Analysis', fontsize=16, fontweight='bold')
    
    frame_numbers = range(len(frame_detections))
    
    # 1. Detection count timeline
    axes[0, 0].plot(frame_numbers, frame_detections, linewidth=1, alpha=0.7, color='blue', label='Raw Counts')
    
    # Add moving average
    window_size = min(30, len(frame_detections) // 10)
    if window_size > 1:
        moving_avg = pd.Series(frame_detections).rolling(window=window_size, center=True).mean()
        axes[0, 0].plot(frame_numbers, moving_avg, linewidth=2, color='red', 
                       label=f'Moving Average ({window_size} frames)')
        axes[0, 0].legend()
    
    axes[0, 0].set_xlabel('Frame Number')
    axes[0, 0].set_ylabel('Detection Count')
    axes[0, 0].set_title('Detection Count Over Time')
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Detection count distribution
    axes[0, 1].hist(frame_detections, bins=min(30, max(frame_detections)), 
                   alpha=0.7, color='green', edgecolor='black')
    mean_detections = np.mean(frame_detections)
    axes[0, 1].axvline(mean_detections, color='red', linestyle='--', 
                      label=f'Mean: {mean_detections:.1f}')
    axes[0, 1].set_xlabel('Detections per Frame')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title('Distribution of Detection Counts')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Detection by object type over time
    colors = plt.cm.Set2(np.linspace(0, 1, len(detection_counts_by_type)))
    for i, (obj_type, counts) in enumerate(detection_counts_by_type.items()):
        if sum(counts) > 0:  # Only plot types with detections
            if len(counts) > 10:
                smoothed = pd.Series(counts).rolling(window=max(5, len(counts)//20), center=True).mean()
                axes[1, 0].plot(range(len(counts)), smoothed, linewidth=2, 
                               color=colors[i], label=f'{obj_type.capitalize()} (total: {sum(counts)})')
            else:
                axes[1, 0].plot(range(len(counts)), counts, linewidth=2, 
                               color=colors[i], label=f'{obj_type.capitalize()} (total: {sum(counts)})')
    
    axes[1, 0].set_xlabel('Frame Number')
    axes[1, 0].set_ylabel('Detection Count')
    axes[1, 0].set_title('Detection Count by Object Type Over Time')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Average confidence over time
    frame_confidences = []
    for frame_idx in range(len(frames_data)):
        frame_confs = [det['confidence'] for det in detection_metrics['all_detections'] 
                      if det['frame_idx'] == frame_idx]
        avg_conf = np.mean(frame_confs) if frame_confs else 0
        frame_confidences.append(avg_conf)
    
    axes[1, 1].plot(range(len(frame_confidences)), frame_confidences, 
                   linewidth=1, alpha=0.7, color='purple', label='Raw Confidence')
    
    # Add moving average for confidence
    if len(frame_confidences) > 10:
        conf_moving_avg = pd.Series(frame_confidences).rolling(window=window_size, center=True).mean()
        axes[1, 1].plot(range(len(frame_confidences)), conf_moving_avg, 
                       linewidth=2, color='orange', label=f'Moving Average ({window_size} frames)')
        axes[1, 1].legend()
    
    axes[1, 1].set_xlabel('Frame Number')
    axes[1, 1].set_ylabel('Average Confidence')
    axes[1, 1].set_title('Average Confidence Over Time')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Temporal statistics
    print("⏱️  TEMPORAL DETECTION STATISTICS:")
    variance = np.var(frame_detections)
    std_dev = np.std(frame_detections)
    coeff_var = std_dev / np.mean(frame_detections) if np.mean(frame_detections) > 0 else 0
    
    print(f"   Detection count variance: {variance:.2f}")
    print(f"   Detection count std deviation: {std_dev:.2f}")
    print(f"   Coefficient of variation: {coeff_var:.3f}")
    
    # Stability assessment
    if coeff_var < 0.2:
        stability = "Excellent"
    elif coeff_var < 0.4:
        stability = "Good"
    elif coeff_var < 0.6:
        stability = "Moderate"
    else:
        stability = "Poor"
    
    print(f"   Stability assessment: {stability}")
    
    # Find outlier frames
    mean_detections = np.mean(frame_detections)
    std_detections = np.std(frame_detections)
    outlier_threshold = 2 * std_detections
    
    high_outliers = [i for i, count in enumerate(frame_detections) 
                    if count > mean_detections + outlier_threshold]
    low_outliers = [i for i, count in enumerate(frame_detections) 
                   if count < max(0, mean_detections - outlier_threshold)]
    
    print(f"   High outlier frames: {len(high_outliers)} frames")
    print(f"   Low outlier frames: {len(low_outliers)} frames")
    
    if high_outliers:
        print(f"   High outlier examples: {high_outliers[:10]}{'...' if len(high_outliers) > 10 else ''}")
    if low_outliers:
        print(f"   Low outlier examples: {low_outliers[:10]}{'...' if len(low_outliers) > 10 else ''}")
        
else:
    print("❌ No data available for temporal analysis")

<a id="spatial"></a>
## 2.4 🗺️ Spatial Distribution Analysis

Understanding where objects are detected reveals field coverage, blind spots, and potential false positive patterns.

### 🎯 Analysis Components:
- **Scatter Plot**: Overall detection distribution across the field
- **Heatmap**: Density visualization showing activity hotspots
- **Object Type Mapping**: Where different objects typically appear
- **Size Analysis**: Consistency of bounding box dimensions

In [ ]:
# Spatial Distribution Analysis
if frames_data and detection_metrics:
    bbox_centers = detection_metrics['bbox_centers']
    spatial_distribution = detection_metrics['spatial_distribution']
    bbox_areas = detection_metrics['bbox_areas']
    
    if bbox_centers:
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('Spatial Distribution Analysis', fontsize=16, fontweight='bold')
        
        x_coords = [center[0] for center in bbox_centers]
        y_coords = [center[1] for center in bbox_centers]
        
        # 1. Overall spatial distribution scatter plot
        axes[0, 0].scatter(x_coords, y_coords, alpha=0.1, s=1, color='blue')
        axes[0, 0].set_xlabel('X Coordinate')
        axes[0, 0].set_ylabel('Y Coordinate')
        axes[0, 0].set_title('Overall Detection Spatial Distribution')
        axes[0, 0].invert_yaxis()  # Invert Y axis to match image coordinates
        axes[0, 0].grid(True, alpha=0.3)
        
        # 2. Detection density heatmap
        try:
            hist, xedges, yedges = np.histogram2d(x_coords, y_coords, bins=50)
            im = axes[0, 1].imshow(hist.T, origin='lower', 
                                  extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]], 
                                  cmap='hot', aspect='auto')
            axes[0, 1].set_xlabel('X Coordinate')
            axes[0, 1].set_ylabel('Y Coordinate')
            axes[0, 1].set_title('Detection Density Heatmap')
            plt.colorbar(im, ax=axes[0, 1], label='Detection Count')
        except Exception as e:
            axes[0, 1].text(0.5, 0.5, f'Heatmap creation failed:\n{str(e)}', 
                           transform=axes[0, 1].transAxes, ha='center', va='center')
            axes[0, 1].set_title('Detection Density Heatmap (Failed)')
        
        # 3. Spatial distribution by object type
        colors = plt.cm.Set1(np.linspace(0, 1, len(spatial_distribution)))
        for i, (obj_type, positions) in enumerate(spatial_distribution.items()):
            if positions:
                x_pos = [pos[0] for pos in positions]
                y_pos = [pos[1] for pos in positions]
                color = colors[i % len(colors)]
                axes[1, 0].scatter(x_pos, y_pos, alpha=0.6, s=2, 
                                  label=f'{obj_type.capitalize()} (n={len(positions)})', 
                                  color=color)
        
        axes[1, 0].set_xlabel('X Coordinate')
        axes[1, 0].set_ylabel('Y Coordinate')
        axes[1, 0].set_title('Spatial Distribution by Object Type')
        axes[1, 0].invert_yaxis()
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        # 4. Bounding box size distribution
        if bbox_areas:
            axes[1, 1].hist(bbox_areas, bins=50, alpha=0.7, color='cyan', edgecolor='black')
            axes[1, 1].set_xlabel('Bounding Box Area (pixels²)')
            axes[1, 1].set_ylabel('Frequency')
            axes[1, 1].set_title('Distribution of Bounding Box Sizes')
            axes[1, 1].set_yscale('log')  # Log scale for better visualization
            axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Spatial statistics
        print("🗺️  SPATIAL DISTRIBUTION STATISTICS:")
        print(f"   X coordinate range: {min(x_coords):.1f} to {max(x_coords):.1f}")
        print(f"   Y coordinate range: {min(y_coords):.1f} to {max(y_coords):.1f}")
        print(f"   X coordinate center: {np.mean(x_coords):.1f} ± {np.std(x_coords):.1f}")
        print(f"   Y coordinate center: {np.mean(y_coords):.1f} ± {np.std(y_coords):.1f}")
        
        # Field coverage analysis
        x_range = max(x_coords) - min(x_coords)
        y_range = max(y_coords) - min(y_coords)
        frame_width = max(x_coords)  # Assuming frame starts at 0
        frame_height = max(y_coords)  # Assuming frame starts at 0
        
        x_coverage = (x_range / frame_width) * 100 if frame_width > 0 else 0
        y_coverage = (y_range / frame_height) * 100 if frame_height > 0 else 0
        
        print(f"   Field coverage X: {x_coverage:.1f}%")
        print(f"   Field coverage Y: {y_coverage:.1f}%")
        
        # Detection density by regions (4x3 grid)
        x_bins, y_bins = 4, 3
        x_step = x_range / x_bins if x_range > 0 else 1
        y_step = y_range / y_bins if y_range > 0 else 1
        
        print(f"\n   Detection density by field regions ({x_bins}x{y_bins} grid):")
        for yi in range(y_bins):
            for xi in range(x_bins):
                x_min = min(x_coords) + xi * x_step
                x_max = min(x_coords) + (xi + 1) * x_step
                y_min = min(y_coords) + yi * y_step
                y_max = min(y_coords) + (yi + 1) * y_step
                
                region_detections = sum(1 for x, y in zip(x_coords, y_coords)
                                      if x_min <= x < x_max and y_min <= y < y_max)
                print(f"     Region ({xi+1},{yi+1}): {region_detections} detections")
        
        # Bounding box statistics
        if bbox_areas:
            print(f"\n📦 BOUNDING BOX STATISTICS:")
            print(f"   Mean area: {np.mean(bbox_areas):.1f} pixels²")
            print(f"   Median area: {np.median(bbox_areas):.1f} pixels²")
            print(f"   Area range: {np.min(bbox_areas):.1f} - {np.max(bbox_areas):.1f} pixels²")
            print(f"   Area std deviation: {np.std(bbox_areas):.1f} pixels²")
            
            # Size categories
            small_boxes = sum(1 for area in bbox_areas if area < 1000)
            medium_boxes = sum(1 for area in bbox_areas if 1000 <= area <= 10000)
            large_boxes = sum(1 for area in bbox_areas if area > 10000)
            
            print(f"   Small boxes (<1000px²): {small_boxes} ({small_boxes/len(bbox_areas)*100:.1f}%)")
            print(f"   Medium boxes (1000-10000px²): {medium_boxes} ({medium_boxes/len(bbox_areas)*100:.1f}%)")
            print(f"   Large boxes (>10000px²): {large_boxes} ({large_boxes/len(bbox_areas)*100:.1f}%)")
        
    else:
        print("❌ No spatial data available for analysis")
else:
    print("❌ No data available for spatial analysis")